# Natural Resource Processing, Sample Selection, and Imputation

**Capstone Project — Moody's Ratings**

Takes the NR panel from NB0 and the master economic panel from NB1 and produces the analysis-ready dataset.

1. Loads `intermediary/natural_resources_production_values.csv` (NB0 output) and splits into production/reserves and prices
2. Backfills early-year mineral production gaps using a constant-share method
3. Computes production and reserves values, classifies resources, aggregates to country-year
4. Merges with the master panel, applies sample filters
5. Imputes remaining missing values (linear interpolation capped at 3 years, then KNN)
6. Merges World Bank population data

**Outputs:**
- `intermediary/NRCleanData.csv` (country-resource-year NR data with prices and values)
- `intermediary/master_data_imputed.csv` (master panel after imputation)
- `intermediary/Master.csv` (final panel with population)
- `intermediary/NaturalResource.csv` (NR data with population)


## 0. Setup

In [1]:
import sys, os
if "scripts" not in sys.path: sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
import os
os.makedirs("intermediary", exist_ok=True)
os.makedirs("Graphics/NB3", exist_ok=True)

import pandas as pd
import numpy as np
from itertools import product
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from standardize_country import add_iso3, ALIAS_TO_ISO3, ISO3_TO_WB


---

# Part A: Load and Prepare Natural Resource Data

## 1. Load NB0 Output

The extraction pipeline (`0_NR_extraction_FINAL.ipynb`) produces a single file with columns:
`Country, Year, Resource, Metric, Value`

Metric includes: Production, Consumption, Reserves, Price.
All mineral volumes are in **kt** (thousand tonnes). Oil is in **kbd**, Coal in **Mt**, Gas in **bcm**.
Prices are in **$/tonne** for minerals, **$/bbl** for oil, **$/tonne** for coal.

In [2]:
nrf = pd.read_csv("intermediary/natural_resources_production_values.csv")  # fix: use NB0 output (includes gas prices)

print(f"Loaded: {nrf.shape[0]:,} rows")
print(f"Resources: {sorted(nrf['Resource'].unique())}")
print(f"Metrics: {sorted(nrf['Metric'].unique())}")
print(f"Year range: {nrf['Year'].min()}-{nrf['Year'].max()}")
print(f"Countries: {nrf['Country'].nunique()}")

price_resources = nrf[nrf['Metric'] == 'Price']['Resource'].unique()
print(f"\nResources with prices: {sorted(price_resources)}")

all_resources = nrf[nrf['Metric'] == 'Production']['Resource'].unique()
missing_prices = set(all_resources) - set(price_resources)
print(f"Resources WITHOUT prices: {sorted(missing_prices)}")

Loaded: 53,978 rows
Resources: ['Aluminium', 'Bauxite', 'Cadmium', 'Coal', 'Cobalt', 'Copper', 'Gold', 'Iron ore', 'Lead', 'Lithium', 'Magnesium compounds', 'Manganese', 'Natural Gas', 'Natural Graphite', 'Nickel', 'Oil', 'Platinum Group', 'Rare Earth', 'Silver', 'Tin', 'Vanadium', 'Zinc']
Metrics: ['Consumption', 'Price', 'Production', 'Reserves']
Year range: 1861-2024
Countries: 177

Resources with prices: ['Aluminium', 'Bauxite', 'Cadmium', 'Coal', 'Cobalt', 'Copper', 'Gold', 'Iron ore', 'Lead', 'Lithium', 'Magnesium compounds', 'Manganese', 'Natural Gas', 'Natural Graphite', 'Nickel', 'Oil', 'Platinum Group', 'Rare Earth', 'Silver', 'Tin', 'Vanadium', 'Zinc']
Resources WITHOUT prices: []


## 2. Check Natural Gas Price Coverage

Gas prices are loaded from NB0's output. This step verifies coverage and flags any remaining gaps before the production value computation.


In [3]:
# DISABLED: NB0 now provides gas prices via ConsolidatedPrices.
# Running this cell would overwrite them with the single-benchmark EI Excel series.
if False:
    # ── Load Natural Gas prices from EI Excel ──
    GAS_PRICE_SHEET = 'Gas - Prices '   # trailing space in EI sheet name

    RAW = "rawdata"
    ei_path = os.path.join(RAW, "base_dataset.xlsx")

    try:
        xl = pd.ExcelFile(ei_path)
        # Find the gas price sheet (handle trailing spaces)
        gas_sheets = [s for s in xl.sheet_names if 'gas' in s.lower() and 'price' in s.lower()]
        print(f"Gas price sheets found: {gas_sheets}")

        if gas_sheets:
            df_gas = pd.read_excel(xl, sheet_name=gas_sheets[0], header=None)
            # EI gas price layout: Year in col 0, price columns from col 1 onward
            # Row 3 typically has headers, data from row 4
            print(f"Sheet shape: {df_gas.shape}")
            print(f"Headers (row 3): {df_gas.iloc[3].tolist()[:6]}")

            # Extract year and first price column (typically Henry Hub or similar benchmark)
            gas_prices = df_gas.iloc[4:, [0, 1]].copy()
            gas_prices.columns = ['Year', 'Value']
            gas_prices['Year'] = pd.to_numeric(gas_prices['Year'], errors='coerce')
            gas_prices['Value'] = pd.to_numeric(gas_prices['Value'], errors='coerce')
            gas_prices = gas_prices.dropna()
            gas_prices['Country'] = 'World'
            gas_prices['Resource'] = 'Natural Gas'
            gas_prices['Metric'] = 'Price'

            # Append to main dataset
            nrf = pd.concat([nrf, gas_prices[['Country', 'Year', 'Value', 'Resource', 'Metric']]], 
                            ignore_index=True)
            print(f"\nGas prices added: {len(gas_prices)} rows, years {int(gas_prices['Year'].min())}-{int(gas_prices['Year'].max())}")
        else:
            print("WARNING: No gas price sheet found in EI Excel. Gas prices remain missing.")

    except FileNotFoundError:
        print(f"WARNING: {ei_path} not found. Gas prices remain missing.")
        print("To fix: ensure base_dataset.xlsx is in the rawdata/ directory.")


## 3. Split into Production/Reserves and Prices

We separate the data into:
- **Country-level production and reserves** (filtered to 1995 onward, excluding Consumption)
- **World-level reference totals** (for the backfilling procedure)
- **Price series** (global prices per resource-year)

In [4]:
nrp = nrf[
    (nrf['Year'] >= 1995)
    & (nrf['Metric'].isin(['Production', 'Reserves']))
][['Country', 'Metric', 'Year', 'Resource', 'Value']].copy()

nrp_world = nrp[nrp['Country'].isin(['World', 'Global'])].copy()

nrp_country = nrp[~nrp['Country'].isin(['World', 'Global'])].copy()

prices = nrf[
    (nrf['Metric'] == 'Price') & (nrf['Year'] >= 1995)
][['Year', 'Resource', 'Value']].copy()
prices.rename(columns={'Value': 'Price'}, inplace=True)
prices = prices.drop_duplicates(subset=['Year', 'Resource'])

print(f"Country-level rows: {len(nrp_country):,}")
print(f"World reference rows: {len(nrp_world):,}")
print(f"Price observations: {len(prices):,}")
print(f"Resources with prices: {prices['Resource'].nunique()}")

Country-level rows: 21,188
World reference rows: 634
Price observations: 620
Resources with prices: 22


## 4. Merge with Prices and Standardise Country Names

Production volumes are joined with global prices. Country names are standardised to World Bank conventions, and non-country entities (OPEC, regional aggregates) are removed.

In [5]:
nrpv = pd.concat([nrp_country, nrp_world], ignore_index=True)

nrpv = pd.merge(nrpv, prices, on=['Year', 'Resource'], how='left')

# Remove resources with poor coverage or duplicates
# Petroleum overlaps with Oil; Platinum Group has coverage issues (see NB1 conflict docs)
nrpv = nrpv[~nrpv['Resource'].isin(['Petroleum', 'Platinum Group'])]

# ── Standardise country names via shared mapping ──
nrpv = add_iso3(nrpv, name_col='Country', iso3_col='Country Code')

# Remove non-country entities (those that didn't map to an ISO3 code)
# but keep World/Global for reference totals
is_ref = nrpv['Country'].isin(['World', 'Global'])
unmatched = nrpv[nrpv['Country Code'].isna() & ~is_ref]['Country'].unique()
if len(unmatched) > 0:
    print(f"WARNING: {len(unmatched)} unmatched country names dropped: {sorted(unmatched)[:10]}")
nrpv = nrpv[nrpv['Country Code'].notna() | is_ref]
nrpv = nrpv.drop_duplicates(subset=['Country', 'Year', 'Resource', 'Metric'])

# Add WB canonical country names
nrpv['Country Name'] = nrpv['Country Code'].map(ISO3_TO_WB)

print(f"Merged NR data: {nrpv.shape[0]:,} rows")
print(f"Resources: {sorted(nrpv['Resource'].unique())}")
print(f"Countries: {nrpv['Country Code'].nunique()}")

# Check price coverage after merge
no_price = nrpv[(nrpv['Metric'] == 'Production') & (nrpv['Price'].isna())]['Resource'].unique()
if len(no_price) > 0:
    print(f"\nWARNING: Resources with production but no price: {sorted(no_price)}")
else:
    print("\nAll production rows have matching prices.")


Merged NR data: 20,590 rows
Resources: ['Aluminium', 'Bauxite', 'Cadmium', 'Coal', 'Cobalt', 'Copper', 'Gold', 'Iron ore', 'Lead', 'Lithium', 'Magnesium compounds', 'Manganese', 'Natural Gas', 'Natural Graphite', 'Nickel', 'Oil', 'Rare Earth', 'Silver', 'Tin', 'Vanadium', 'Zinc']
Countries: 156



## 5. Backfill Production Gaps for Selected Minerals

Some minerals have incomplete country-level coverage in early years. For each mineral, we identify the earliest year where country-level data accounts for over 70% of global production. We then backfill earlier years by applying each country's production share at that threshold year to the known global totals.

This constant-share assumption is a simplification, but it preserves cross-country relativities and avoids introducing zeros for countries that were producing but not reporting.

In [6]:
target_minerals = ["Tin", "Nickel", "Vanadium", "Manganese", "Cadmium",
                   "Magnesium compounds", "Iron ore"]

prod = nrpv[nrpv['Metric'] == 'Production'].copy()

global_prod = prod[prod['Country'] == 'Global'][['Year', 'Resource', 'Value']].copy()
global_prod.rename(columns={'Value': 'Global_Production'}, inplace=True)

world_prod = prod[prod['Country'] == 'World'][['Year', 'Resource', 'Value']].copy()
world_prod.rename(columns={'Value': 'World_Production'}, inplace=True)

reference_prod = pd.merge(global_prod, world_prod, on=['Year', 'Resource'], how='outer')
reference_prod['Reference_Production'] = (
    reference_prod['Global_Production'].fillna(reference_prod['World_Production'])
)

country_prod = prod[~prod['Country'].isin(['World', 'Global'])].copy()
country_with_ref = pd.merge(
    country_prod[['Country', 'Year', 'Resource', 'Value']],
    reference_prod[['Year', 'Resource', 'Reference_Production']],
    on=['Year', 'Resource'], how='left',
)
country_with_ref['Country_Share'] = (
    country_with_ref['Value'] / country_with_ref['Reference_Production']
)

# Total country coverage per resource-year
country_total = country_prod.groupby(['Year', 'Resource'])['Value'].sum().reset_index()
country_total.rename(columns={'Value': 'Country_Total'}, inplace=True)
country_total = pd.merge(
    country_total, reference_prod[['Year', 'Resource', 'Reference_Production']],
    on=['Year', 'Resource'], how='left',
)
country_total['Coverage'] = (
    country_total['Country_Total'] / country_total['Reference_Production']
) * 100

threshold_years = {}
constant_shares = {}

for mineral in target_minerals:
    mineral_coverage = country_total[
        (country_total['Resource'] == mineral) & (country_total['Coverage'] > 70)
    ].sort_values('Year')

    if len(mineral_coverage) > 0:
        threshold_year = mineral_coverage['Year'].min()
        threshold_years[mineral] = threshold_year

        shares_at_threshold = country_with_ref[
            (country_with_ref['Resource'] == mineral)
            & (country_with_ref['Year'] == threshold_year)
            & (country_with_ref['Value'] > 0)
        ][['Country', 'Country_Share']].copy()
        constant_shares[mineral] = shares_at_threshold
    else:
        threshold_years[mineral] = None
        print(f"WARNING: {mineral} never reaches 70% coverage")

print("Threshold years (earliest year with >70% country coverage):")
for mineral, year in threshold_years.items():
    n_countries = len(constant_shares.get(mineral, []))
    print(f"  {mineral}: {year} ({n_countries} producing countries)")

Threshold years (earliest year with >70% country coverage):
  Tin: 2014 (7 producing countries)
  Nickel: 2013 (7 producing countries)
  Vanadium: 2014 (4 producing countries)
  Manganese: 2014 (6 producing countries)
  Cadmium: 2020 (14 producing countries)
  Magnesium compounds: 2020 (10 producing countries)
  Iron ore: 2020 (16 producing countries)


In [7]:
backfilled_rows = []

for mineral, threshold_year in threshold_years.items():
    if threshold_year is None:
        continue

    years_to_backfill = sorted([y for y in prod['Year'].unique() if y < threshold_year])
    ref_for_backfill = reference_prod[
        (reference_prod['Resource'] == mineral)
        & (reference_prod['Year'].isin(years_to_backfill))
    ][['Year', 'Reference_Production']].copy()

    shares = constant_shares[mineral]

    for _, ref_row in ref_for_backfill.iterrows():
        year = ref_row['Year']
        ref_prod_value = ref_row['Reference_Production']
        if pd.isna(ref_prod_value):
            continue

        for _, share_row in shares.iterrows():
            backfilled_rows.append({
                'Country': share_row['Country'],
                'Year': year,
                'Resource': mineral,
                'Value': ref_prod_value * share_row['Country_Share'],
                'Metric': 'Production',
                'Source': 'Backfilled',
            })

backfilled_df = pd.DataFrame(backfilled_rows)
print(f"Backfilled rows created: {len(backfilled_df):,}")
print(f"\nBy mineral:")
print(backfilled_df.groupby('Resource').agg(
    Years=('Year', lambda x: f"{x.min()}-{x.max()}"),
    Countries=('Country', 'nunique'),
    Rows=('Year', 'count'),
))

nrpv_clean = nrpv[~nrpv['Country'].isin(['World', 'Global'])].copy()
nrpv_clean['Source'] = 'Original'

backfilled_for_merge = backfilled_df.copy()
backfilled_for_merge = pd.merge(
    backfilled_for_merge, prices,
    on=['Year', 'Resource'], how='left',
)

# ── FIX (Check 8): restore Country Code / Country Name on backfilled rows ──
# backfilled_df only carries ['Country','Year','Resource','Value','Metric','Source'];
# the price merge adds 'Price' but not the ISO3 columns added in cell 10.
# Without this block the column selector two lines below raises a KeyError.
backfilled_for_merge = add_iso3(backfilled_for_merge, name_col='Country', iso3_col='Country Code')
backfilled_for_merge['Country Name'] = backfilled_for_merge['Country Code'].map(ISO3_TO_WB)
backfilled_for_merge = backfilled_for_merge[list(nrpv_clean.columns)]

# Remove original partial data for target minerals before threshold year
for mineral, threshold_year in threshold_years.items():
    if threshold_year is None:
        continue
    mask = ((nrpv_clean['Resource'] == mineral)
            & (nrpv_clean['Year'] < threshold_year)
            & (nrpv_clean['Metric'] == 'Production'))
    nrpv_clean = nrpv_clean[~mask]

nrpv = pd.concat([nrpv_clean, backfilled_for_merge], ignore_index=True)
nrpv = nrpv.sort_values(['Resource', 'Country', 'Year', 'Metric']).reset_index(drop=True)

print(f"\nFinal NR dataset: {nrpv.shape[0]:,} rows")

Backfilled rows created: 1,437

By mineral:
                             Years  Countries  Rows
Resource                                           
Cadmium              1995.0-2019.0         14   350
Iron ore             1995.0-2019.0         16   400
Magnesium compounds  1995.0-2019.0         10   250
Manganese            1995.0-2013.0          6   114
Nickel               1995.0-2012.0          7   126
Tin                  1995.0-2013.0          7   133
Vanadium             1995.0-2013.0          4    64

Final NR dataset: 21,295 rows


### 5a. Backfill Sensitivity Analysis

The 70% coverage threshold that defines the backfill anchor year is a modelling choice. This diagnostic compares three thresholds (60%, 70%, 80%) to check how sensitive the backfilled values are to that choice. A mineral whose threshold year shifts substantially across thresholds has less reliable backfilled data in early years.

The constant-share assumption (projecting the threshold year's country shares backward) is a second source of uncertainty. If a mineral's production was more geographically concentrated in earlier years, the backfill will understate dominant producers and overstate minor ones. This is inherent to the method and cannot be resolved without additional historical data.


In [8]:
# Compares total backfilled production under three coverage thresholds.
# Diagnostic only — does not modify the dataset.

THRESHOLDS = [60, 70, 80]
sensitivity_rows = []

for threshold_pct in THRESHOLDS:
    for mineral in target_minerals:
        mineral_coverage = country_total[
            (country_total['Resource'] == mineral) & (country_total['Coverage'] > threshold_pct)
        ].sort_values('Year')

        if len(mineral_coverage) == 0:
            sensitivity_rows.append({
                'Threshold': f'{threshold_pct}%',
                'Mineral': mineral,
                'Threshold_Year': None,
                'N_Countries': 0,
                'Years_Backfilled': 0,
                'Total_Backfilled_Production': 0,
                'Max_Country_Share': None,
                'Top_Country': None,
            })
            continue

        t_year = mineral_coverage['Year'].min()
        shares_at_t = country_with_ref[
            (country_with_ref['Resource'] == mineral)
            & (country_with_ref['Year'] == t_year)
            & (country_with_ref['Value'] > 0)
        ][['Country', 'Country_Share']].copy()

        years_to_bf = [y for y in prod['Year'].unique() if y < t_year]
        ref_subset = reference_prod[
            (reference_prod['Resource'] == mineral)
            & (reference_prod['Year'].isin(years_to_bf))
        ]

        total_bf = 0
        for _, ref_row in ref_subset.iterrows():
            if pd.notna(ref_row['Reference_Production']):
                total_bf += ref_row['Reference_Production'] * shares_at_t['Country_Share'].sum()

        top_country = shares_at_t.loc[shares_at_t['Country_Share'].idxmax()]

        sensitivity_rows.append({
            'Threshold': f'{threshold_pct}%',
            'Mineral': mineral,
            'Threshold_Year': int(t_year),
            'N_Countries': len(shares_at_t),
            'Years_Backfilled': len(years_to_bf),
            'Total_Backfilled_Production': round(total_bf, 2),
            'Max_Country_Share': round(top_country['Country_Share'] * 100, 1),
            'Top_Country': top_country['Country'],
        })

sens_df = pd.DataFrame(sensitivity_rows)

print("=" * 90)
print("BACKFILL SENSITIVITY: How threshold choice affects backfilled data")
print("=" * 90)
for mineral in target_minerals:
    sub = sens_df[sens_df['Mineral'] == mineral]
    print()
    print(f"  {mineral}:")
    for _, row in sub.iterrows():
        if row['Threshold_Year'] is None:
            print(f"    {row['Threshold']}: never reaches threshold")
        else:
            print(f"    {row['Threshold']}: threshold year {row['Threshold_Year']}, "
                  f"{row['Years_Backfilled']} years backfilled, "
                  f"{row['N_Countries']} countries, "
                  f"top producer {row['Top_Country']} ({row['Max_Country_Share']}%)")

# Flag minerals where the threshold year shifts by 3+ years across thresholds
print()
print("  Minerals sensitive to threshold choice (threshold year shifts >= 3 years):")
any_sensitive = False
for mineral in target_minerals:
    sub = sens_df[(sens_df['Mineral'] == mineral) & (sens_df['Threshold_Year'].notna())]
    if len(sub) >= 2:
        year_range = sub['Threshold_Year'].max() - sub['Threshold_Year'].min()
        if year_range >= 3:
            any_sensitive = True
            print(f"    {mineral}: threshold year ranges from "
                  f"{int(sub['Threshold_Year'].min())} to {int(sub['Threshold_Year'].max())} "
                  f"(delta = {int(year_range)} years)")
if not any_sensitive:
    print("    None (all minerals are stable across thresholds).")


BACKFILL SENSITIVITY: How threshold choice affects backfilled data

  Tin:
    60%: threshold year 2014, 19 years backfilled, 7 countries, top producer China (29.8%)
    70%: threshold year 2014, 19 years backfilled, 7 countries, top producer China (29.8%)
    80%: threshold year 2015, 20 years backfilled, 7 countries, top producer China (35.8%)

  Nickel:
    60%: threshold year 2013, 18 years backfilled, 7 countries, top producer Philippines (17.1%)
    70%: threshold year 2013, 18 years backfilled, 7 countries, top producer Philippines (17.1%)
    80%: threshold year 2015, 20 years backfilled, 7 countries, top producer Philippines (25.1%)

  Vanadium:
    60%: threshold year 2014, 19 years backfilled, 4 countries, top producer China (49.0%)
    70%: threshold year 2014, 19 years backfilled, 4 countries, top producer China (49.0%)
    80%: threshold year 2014, 19 years backfilled, 4 countries, top producer China (49.0%)

  Manganese:
    60%: threshold year 2014, 19 years backfilled,

## 6. Price Gap Check (2021)

Checks for missing 2021 prices. If any resource has no 2021 price in the NB0 output, a fallback based on prior-year values is applied. Review this cell if the consolidated price file is updated to cover 2021.


In [9]:
price_changes_2021 = {
    "Copper": 44.82477588,       # Base Metals (IEA)
    "Nickel": 50.13077594,       # Battery Metals (IEA)
    "Rare Earth": 90.52167524,   # Rare Earth (IEA) — NOTE: verify against IEA source; documentation says 108.6%
}

for resource, pct_change in price_changes_2021.items():
    mask_2021 = (nrpv['Resource'] == resource) & (nrpv['Year'] == 2021)
    mask_2020 = (nrpv['Resource'] == resource) & (nrpv['Year'] == 2020)

    price_2020 = nrpv.loc[mask_2020, 'Price'].values
    if len(price_2020) > 0:
        price_2021 = price_2020[0] * (1 + pct_change / 100)
        nrpv.loc[mask_2021, 'Price'] = price_2021
        print(f"{resource}: 2020 price = {price_2020[0]:,.2f}, "
              f"YoY = +{pct_change:.1f}%, 2021 price = {price_2021:,.2f}")

Copper: 2020 price = 6,320.00, YoY = +44.8%, 2021 price = 9,152.93
Nickel: 2020 price = 16,634.73, YoY = +50.1%, 2021 price = 24,973.86
Rare Earth: 2020 price = 5,130.00, YoY = +90.5%, 2021 price = 9,773.76


## 7. Compute Production and Reserves Values

With prices attached, we compute total production value (Volume x Price) and total reserves value for each country-resource-year observation. Resources are classified into three categories: Hydrocarbons, Subsoil Metals, and Precious Metals.

In [10]:
# Filter to pre-2022 and remove remaining problematic resources
nrpv = nrpv[
    (nrpv['Year'] < 2022)
    & (~nrpv['Resource'].isin(['Petroleum', 'Platinum Group']))
]

# Pivot to have Production and Reserves as columns
nrpv = nrpv.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Resource', 'Source', 'Price'],  # FIX: Country Code must be in index or pivot drops it
    columns='Metric', values='Value',
).reset_index()

# Guard: pivot_table only creates 'Reserves' if ≥1 row has Metric='Reserves'.
# Resources with production-only data (e.g. backfilled minerals) will be missing
# the column — fill with NaN so downstream aggregations treat them as no-reserves.
if 'Production' not in nrpv.columns:
    nrpv['Production'] = float('nan')
if 'Reserves' not in nrpv.columns:
    nrpv['Reserves'] = float('nan')


# Annualise production values before computing dollar totals.
# Oil production is stored in kbd (thousand barrels per day).
#   kbd × $/bbl = $(thousands)/day  →  multiply by 1,000 × 365 to get annual USD.
# Gas prices loaded from EI Excel are in $/MMBtu; volumes are in bcm/year.
#   1 bcm = 35,314,667 MMBtu  →  multiply production by that factor.
# Minerals and coal (kt or Mt × $/tonne) are already on an annual basis.
PRODUCTION_UNIT_SCALE = {
    'Oil':         1_000 * 365,   # kbd → barrels/year
    'Natural Gas': 35_315_000 * 1.037,  # bcm -> MMBtu/year (35.315bn cf/bcm; 1 Mcf ~ 1.037 MMBtu — matches NB0 PRODVAL_MULTIPLIER)
    'Coal':        1_000_000,     # Mt → tonnes
}
MINERAL_PRODVAL_DEFAULT = 1_000   # kt → tonnes — use as fillna instead of 1.0
_scale = nrpv['Resource'].map(PRODUCTION_UNIT_SCALE).fillna(MINERAL_PRODVAL_DEFAULT)
nrpv['Production_TotalValue'] = nrpv['Production'] * nrpv['Price'] * _scale
nrpv['Reserves_TotalValue'] = nrpv['Reserves'] * nrpv['Price']  # reserves not annualised

hydrocarbons = ['Oil', 'Natural Gas', 'Coal']
precious_metals = ['Gold', 'Silver']
subsoil = [
    'Bauxite', 'Copper', 'Aluminium', 'Lead', 'Lithium',
    'Zinc', 'Cadmium', 'Cobalt', 'Iron ore', 'Magnesium compounds',
    'Manganese', 'Nickel', 'Rare Earth', 'Tin', 'Vanadium',
    'Natural Graphite',
]

def classify_resource(x):
    if x in hydrocarbons:
        return 'Hydrocarbons'
    elif x in subsoil:
        return 'Subsoil Metals'
    elif x in precious_metals:
        return 'Precious Metals'
    else:
        return 'Others'

nrpv['Resource Category'] = nrpv['Resource'].apply(classify_resource)
print(f"NR with values: {nrpv.shape[0]:,} rows")
print(f"Resource categories: {nrpv['Resource Category'].value_counts().to_dict()}")

NR with values: 18,912 rows
Resource categories: {'Subsoil Metals': 8834, 'Hydrocarbons': 5817, 'Precious Metals': 4261}


## 8. Aggregate to Country-Year Level

Production values are summed by resource category and country-year. Dominance dummies are created: a country is classified as hydrocarbon-dominant, subsoil-metal-dominant, or precious-metal-dominant if that category accounts for more than 50% of its total production value in a given year.

In [11]:
# Total production value by country, year, resource category
category_totals = (
    nrpv.groupby(['Country', 'Country Code', 'Year', 'Resource Category'])['Production_TotalValue']
    .sum().reset_index()
)

# Total across all categories
country_year_totals = (
    nrpv.groupby(['Country', 'Country Code', 'Year'])['Production_TotalValue']
    .sum().reset_index()
    .rename(columns={'Production_TotalValue': 'Total_Production_Value'})
)

# Pivot categories to columns
category_pivot = category_totals.pivot_table(
    index=['Country', 'Country Code', 'Year'], columns='Resource Category',
    values='Production_TotalValue', fill_value=0,
).reset_index()

merged = category_pivot.merge(country_year_totals, on=['Country', 'Country Code', 'Year'])

# Dominance dummies (>50% share)
# FIX (Check 7): guard against division by zero when Total_Production_Value == 0.
# Pandas would silently produce NaN > 0.5 → False → 0, but the intent should be explicit.
_tpv = merged['Total_Production_Value']
merged['Hydrocarbons_Dominant'] = np.where(
    _tpv > 0, (merged.get('Hydrocarbons', 0) / _tpv > 0.5).astype(int), 0)
merged['Subsoil_Metals_Dominant'] = np.where(
    _tpv > 0, (merged.get('Subsoil Metals', 0) / _tpv > 0.5).astype(int), 0)
merged['Precious_Metals_Dominant'] = np.where(
    _tpv > 0, (merged.get('Precious Metals', 0) / _tpv > 0.5).astype(int), 0)

# Aggregate reserves and total production quantities
reserves_totals = nrpv.groupby(['Country', 'Country Code', 'Year']).agg({
    'Reserves_TotalValue': 'sum',
    'Reserves': 'sum',
    'Production': 'sum',
}).reset_index().rename(columns={
    'Reserves_TotalValue': 'Total_Reserves_Value',
    'Reserves': 'Total_Reserves',
    'Production': 'Total_Production',
})

# Final NR aggregated dataset — include dominance columns directly in initial selection
# to avoid fragile index alignment from separate assignment lines
nrpa = merged[['Country', 'Country Code', 'Year', 'Total_Production_Value',
               'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
               'Precious_Metals_Dominant']].merge(
    reserves_totals, on=['Country', 'Country Code', 'Year'], how='left',
)

nrpa = nrpa[['Country', 'Country Code', 'Year', 'Total_Production', 'Total_Reserves',
             'Total_Production_Value', 'Total_Reserves_Value',
             'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
             'Precious_Metals_Dominant']]

feasible_countries = nrpa['Country Code'].unique().tolist()
print(f"NR aggregated data: {nrpa.shape[0]:,} rows")
print(f"Countries with NR data: {nrpa['Country'].nunique()}")

NR aggregated data: 4,023 rows
Countries with NR data: 156


---

# Part B: Sample Selection and Variable Filtering

## 9. Merge with Master Data and Apply Filters

The NR variables are merged with the master panel from Step 1. We then:
1. Select the subset of analysis variables (informed by the missingness diagnostics in Step 2)
2. Restrict to countries present in the NR dataset
3. Exclude countries with structurally excessive missingness (identified iteratively)
4. Fill structural zeros for IMF credit and death rates

In [12]:
df = pd.read_csv("intermediary/master_data_wide.csv")

# Merge with NR aggregated variables (on Country Code to avoid name mismatches)
df = pd.merge(df, nrpa.drop(columns=["Country"]), on=["Country Code", "Year"], how="left")

keep_vars = [
    'Country Code', 'Country Name', 'Year',
    # Governance
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'Agriculture',
    'Capital depreciation rate',
    'Clientelism index',
    'Death rates, crude per 1000 people',
    'Domestic credit to private sector (% of GDP)',
    'Economic Complexity Index',
    'GDP per capita (constant prices, PPP)',
    'Government revenue',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Human capital index',
    'Industry',
    'Inflation, consumer prices (annual %)',
    'Landlocked',
    'Lending interest rate (%)',
    'Life expectancy at birth, total (years)',
    'Manufacturing',
    'Mineral rents (% of GDP)',
    'Mobile cellular subscriptions (per 100 people)',
    'Natural gas rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Forestry rents (% of GDP)',
    'Political corruption index',
    'Political stability — estimate',
    'Primary net lending, General government, Percent of GDP',
    'Property rights',
    'Real interest rate (%)',
    'Rule of law index',
    'Services',
    'Share of consumption in GDP',
    'Share of government spending in GDP',
    'Share of investment in GDP',
    'Total natural resources rents (% of GDP)',
    'Trade (% of GDP)',
    'Urban population (% of total population)',
    'Use of IMF credit (DOD, current US$)',
    # NR variables
    'Total_Production',
    'Total_Reserves',
    'Total_Production_Value',
    'Total_Reserves_Value',
    'Hydrocarbons_Dominant',
    'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
]

# Based on iterative missingness analysis (see Step 2 diagnostics)
omit_countries = [
    'BDI', 'BTN', 'CAF', 'ERI', 'FJI', 'GUY', 'ISL', 'MNE', 'NCL',
    'PRK', 'SLB', 'SLE', 'SSD', 'SUR', 'SYR', 'GUF', 'TWN', 'CUB',
    'TLS', 'AFG', 'TKM', 'KHM', 'XKX', 'BEN',
]

# Apply filters
df = df[keep_vars]
df = df[df["Country Code"].isin(feasible_countries)]
df = df[~df["Country Code"].isin(omit_countries)]

# Fill structural zeros
df["Use of IMF credit (DOD, current US$)"] = df["Use of IMF credit (DOD, current US$)"].fillna(0)
df["Death rates, crude per 1000 people"] = df["Death rates, crude per 1000 people"].fillna(0)

cmaster = df.copy()

print(f"Filtered sample: {cmaster.shape[0]:,} rows")
print(f"Countries: {cmaster['Country Code'].nunique()}")
print(f"Variables: {cmaster.shape[1] - 3} (excl. identifiers)")
print(f"Remaining missing cells: {cmaster.iloc[:, 3:].isna().sum().sum():,}")

Filtered sample: 3,275 rows
Countries: 131
Variables: 44 (excl. identifiers)
Remaining missing cells: 7,065


---

# Part C: Imputation

## 10. Linear Interpolation (Within-Country)

The first imputation pass uses linear interpolation within each country's time series, with a hard cap on gap length (`MAX_GAP = 3` years). This prevents the imputer from filling long stretches of missing data with flat constants, which would happen when a country's first observation is many years into the panel.

Three types of gap are handled separately:
- **Interior gaps** (between two observed values): filled by linear interpolation if the gap is ≤ 3 consecutive years.
- **Leading gaps** (before the first observation): extrapolated backward only if the first observation is within 3 years of the panel start.
- **Trailing gaps** (after the last observation): extrapolated forward only if the last observation is within 3 years of the panel end.

Gaps exceeding the cap are left as NaN for KNN (Step 11) to handle using cross-country information, which is preferable to carrying a single value flat across a decade.

Variables excluded: `Landlocked` (time-invariant dummy), dominance dummies, and total reserves (which should remain as-is).

In [13]:
EXCLUDE_VARS = [
    'Landlocked',
    'Subsoil_Metals_Dominant',
    'Hydrocarbons_Dominant',
    'Precious_Metals_Dominant',
    'Total_Reserves_Value',
    'Total_Reserves',
]

MAX_GAP = 3   # maximum years to interpolate or extrapolate


def impute_linear_interpolation(df, max_gap=MAX_GAP):
    """
    Linear interpolation within each country's time series, with a hard cap
    on the gap length that can be filled.

    Interior gaps:  filled only if the gap is <= max_gap consecutive years.
    Leading gaps:   filled only if the first observation is <= max_gap years
                    after the panel start (i.e. at most max_gap years backward).
    Trailing gaps:  filled only if the last observation is <= max_gap years
                    before the panel end (i.e. at most max_gap years forward).

    Gaps exceeding the cap are left as NaN for KNN to handle.
    """
    id_cols = ['Country Code', 'Country Name', 'Year']
    data_cols = [c for c in df.columns if c not in id_cols and c not in EXCLUDE_VARS]

    pre_missing = df[data_cols].isna().sum().sum()
    print("=" * 70)
    print(f"IMPUTATION STEP 1: Linear interpolation by country (max_gap={max_gap})")
    print("=" * 70)
    print(f"\nVariables excluded: {[c for c in EXCLUDE_VARS if c in df.columns]}")
    print(f"Variables to impute: {len(data_cols)}")
    print(f"Total missing cells before: {pre_missing:,}")

    df_sorted = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

    # Tracking
    cells_interior = 0
    cells_leading = 0
    cells_trailing = 0
    cells_skipped_too_long = 0

    imputed_dfs = []
    for country in df_sorted['Country Code'].unique():
        country_data = df_sorted[df_sorted['Country Code'] == country].copy()

        for col in data_cols:
            series = country_data[col].values.copy().astype(float)
            observed_mask = ~np.isnan(series)

            if not observed_mask.any():
                continue

            # Indices of first and last observed values
            obs_indices = np.where(observed_mask)[0]
            first_obs = obs_indices[0]
            last_obs = obs_indices[-1]

            # --- Interior interpolation (between observed points, capped) ---
            interior = pd.Series(series)
            interior = interior.interpolate(
                method='linear', limit=max_gap, limit_direction='forward',
                limit_area='inside'
            )

            filled_interior = np.isnan(series) & ~np.isnan(interior.values)
            cells_interior += filled_interior.sum()
            series = interior.values

            # --- Leading extrapolation (before first obs, capped) ---
            if first_obs > 0 and first_obs <= max_gap:
                fill_series = pd.Series(series)
                fill_series = fill_series.interpolate(
                    method='linear', limit=max_gap, limit_direction='backward',
                    limit_area=None
                )
                leading_filled = (
                    np.isnan(series[:first_obs])
                    & ~np.isnan(fill_series.values[:first_obs])
                )
                cells_leading += leading_filled.sum()
                series[:first_obs] = fill_series.values[:first_obs]
            elif first_obs > max_gap:
                cells_skipped_too_long += first_obs

            # --- Trailing extrapolation (after last obs, capped) ---
            n = len(series)
            trailing_gap = n - 1 - last_obs
            if trailing_gap > 0 and trailing_gap <= max_gap:
                fill_series = pd.Series(series)
                fill_series = fill_series.interpolate(
                    method='linear', limit=max_gap, limit_direction='forward',
                    limit_area=None
                )
                trailing_filled = (
                    np.isnan(series[last_obs + 1:])
                    & ~np.isnan(fill_series.values[last_obs + 1:])
                )
                cells_trailing += trailing_filled.sum()
                series[last_obs + 1:] = fill_series.values[last_obs + 1:]
            elif trailing_gap > max_gap:
                cells_skipped_too_long += trailing_gap

            country_data[col] = series

        imputed_dfs.append(country_data)

    df_imputed = pd.concat(imputed_dfs, ignore_index=True)

    post_missing = df_imputed[data_cols].isna().sum().sum()
    total_filled = pre_missing - post_missing

    print(f"\nTotal missing cells after:  {post_missing:,}")
    print(f"Cells filled:              {total_filled:,} "
          f"({100 * total_filled / pre_missing:.1f}%)")
    print(f"  - Interior interpolation: {cells_interior:,}")
    print(f"  - Leading extrapolation:  {cells_leading:,}")
    print(f"  - Trailing extrapolation: {cells_trailing:,}")
    print(f"  - Skipped (gap > {max_gap}yr):  {cells_skipped_too_long:,} cells left for KNN")

    remaining = df_imputed[data_cols].isna().sum()
    remaining = remaining[remaining > 0].sort_values(ascending=False)
    if len(remaining) > 0:
        print(f"\nVariables with remaining missing:")
        for var, count in remaining.items():
            print(f"   {var}: {count} cells")
    else:
        print("\nAll missing values filled.")

    return df_imputed


df_imputed = impute_linear_interpolation(cmaster)


IMPUTATION STEP 1: Linear interpolation by country (max_gap=3)

Variables excluded: ['Landlocked', 'Subsoil_Metals_Dominant', 'Hydrocarbons_Dominant', 'Precious_Metals_Dominant', 'Total_Reserves_Value', 'Total_Reserves']
Variables to impute: 38
Total missing cells before: 6,640

Total missing cells after:  5,707
Cells filled:              933 (14.1%)
  - Interior interpolation: 492
  - Leading extrapolation:  341
  - Trailing extrapolation: 100
  - Skipped (gap > 3yr):  3,041 cells left for KNN

Variables with remaining missing:
   Real interest rate (%): 1087 cells
   Lending interest rate (%): 1087 cells
   Domestic credit to private sector (% of GDP): 628 cells
   Adjusted savings: gross savings (% of GNI): 478 cells
   Human capital index: 300 cells
   Manufacturing: 279 cells
   Trade (% of GDP): 246 cells
   Primary net lending, General government, Percent of GDP: 209 cells
   Inflation, consumer prices (annual %): 149 cells
   Services: 140 cells
   Access to electricity (% of p

## 11. KNN Imputation (Cross-Country)

After linear interpolation, some cells may still be missing if a country has no data at all for a particular variable. These are filled using K-Nearest Neighbours imputation (k=5, distance-weighted), which finds the most similar countries in the feature space and uses their values.

Before applying KNN to the full dataset, we run a validation test: we randomly mask 5% of known values, impute them, and compare predictions to actuals.

In [14]:
# Extended exclusion list for KNN (includes production totals)
EXCLUDE_VARS_KNN = [
    'Landlocked',
    'Subsoil_Metals_Dominant',
    'Hydrocarbons_Dominant',
    'Precious_Metals_Dominant',
    'Total_Production_Value',
    'Total_Reserves_Value',
    'Total_Production',
    'Total_Reserves',
]


def validate_knn_imputer(df, test_fraction=0.05, n_neighbors=5, random_state=42):
    id_cols = ['Country Code', 'Country Name', 'Year']
    numeric_cols = [c for c in df.columns
                    if c not in id_cols and c not in EXCLUDE_VARS_KNN
                    and df[c].dtype in ['float64', 'int64']]

    print("=" * 70)
    print("KNN IMPUTER VALIDATION TEST")
    print("=" * 70)
    print(f"\nVariables to impute: {len(numeric_cols)}")
    print(f"n_neighbors: {n_neighbors}")

    df_numeric = df[numeric_cols].copy()

    # Randomly mask known values
    np.random.seed(random_state)
    non_missing_mask = ~df_numeric.isna()
    non_missing_indices = list(zip(*np.where(non_missing_mask)))
    n_to_mask = int(len(non_missing_indices) * test_fraction)
    test_indices = np.random.choice(len(non_missing_indices), size=n_to_mask, replace=False)
    test_positions = [non_missing_indices[i] for i in test_indices]

    print(f"Total non-missing cells: {len(non_missing_indices):,}")
    print(f"Cells masked for testing: {n_to_mask:,} ({test_fraction * 100:.0f}%)")

    true_values = []
    for row, col in test_positions:
        true_values.append({
            'row': row, 'col': col,
            'variable': numeric_cols[col],
            'true_value': df_numeric.iloc[row, col],
        })

    df_masked = df_numeric.copy()
    for row, col in test_positions:
        df_masked.iloc[row, col] = np.nan

    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df_masked.fillna(df_masked.mean())),
        columns=numeric_cols,
    )
    df_scaled[df_masked.isna()] = np.nan

    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    df_imputed_scaled = imputer.fit_transform(df_scaled)
    df_result = pd.DataFrame(
        scaler.inverse_transform(df_imputed_scaled), columns=numeric_cols,
    )

    for item in true_values:
        item['predicted_value'] = df_result.iloc[item['row'], item['col']]
        item['error'] = item['predicted_value'] - item['true_value']
        item['abs_error'] = abs(item['error'])
        item['pct_error'] = (
            100 * item['abs_error'] / abs(item['true_value'])
            if item['true_value'] != 0 else np.nan
        )

    results_df = pd.DataFrame(true_values)
    y_true = results_df['true_value'].values
    y_pred = results_df['predicted_value'].values

    print(f"\nR2 Score: {r2_score(y_true, y_pred):.4f}")
    print(f"MAE: {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"Median % Error: {results_df['pct_error'].median():.2f}%")

    var_results = results_df.groupby('variable').agg({
        'true_value': 'count',
        'abs_error': 'mean',
        'pct_error': 'median',
    }).rename(columns={
        'true_value': 'n_tested',
        'abs_error': 'mean_abs_error',
        'pct_error': 'median_pct_error',
    }).sort_values('median_pct_error', ascending=False)

    print("\nVariables with HIGHEST error:")
    print(var_results.head(10).to_string())
    print("\nVariables with LOWEST error:")
    print(var_results.tail(10).to_string())

    return results_df, var_results


results_df, var_results = validate_knn_imputer(df_imputed, test_fraction=0.05, n_neighbors=5)

KNN IMPUTER VALIDATION TEST

Variables to impute: 36
n_neighbors: 5
Total non-missing cells: 112,287
Cells masked for testing: 5,614 (5%)

R2 Score: 0.7929
MAE: 7270413.2153
RMSE: 184424713.5208
Median % Error: 4.41%

Variables with HIGHEST error:
                                                         n_tested  mean_abs_error  median_pct_error
variable                                                                                           
Primary net lending, General government, Percent of GDP       172        2.159293         56.481505
Real interest rate (%)                                        102        5.133702         40.051353
Inflation, consumer prices (annual %)                         155        3.817569         38.821737
Mineral rents (% of GDP)                                      147        0.415043         36.905184
Natural gas rents (% of GDP)                                  163        0.180819         29.916479
Oil rents (% of GDP)                                

In [15]:
def apply_knn_imputer(df, n_neighbors=5):
    """
    KNN imputation on remaining missing cells after linear interpolation.

    Returns
    -------
    df_final : DataFrame with imputed values.
    knn_mask : DataFrame (same shape as df[numeric_cols]) where True = cell was filled by KNN.
    """
    id_cols = ['Country Code', 'Country Name', 'Year']
    numeric_cols = [c for c in df.columns
                    if c not in id_cols and c not in EXCLUDE_VARS_KNN
                    and df[c].dtype in ['float64', 'int64']]

    print("=" * 70)
    print("IMPUTATION STEP 2: KNN imputation (cross-country)")
    print("=" * 70)

    pre_missing = df[numeric_cols].isna().sum().sum()
    print()
    print(f"Missing before: {pre_missing:,}")

    # Record which cells are missing before KNN (these are the ones KNN will fill)
    was_missing = df[numeric_cols].isna().copy()

    scaler = StandardScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df[numeric_cols].fillna(df[numeric_cols].mean())),
        columns=numeric_cols,
    )
    df_scaled[df[numeric_cols].isna()] = np.nan

    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    imputed_array = imputer.fit_transform(df_scaled)
    imputed_values = scaler.inverse_transform(imputed_array)

    df_final = df.copy()
    df_final[numeric_cols] = imputed_values

    post_missing = df_final[numeric_cols].isna().sum().sum()
    print(f"Missing after: {post_missing:,}")
    print(f"Filled: {pre_missing - post_missing:,} cells")

    excluded_in_data = [c for c in EXCLUDE_VARS_KNN if c in df.columns]
    if excluded_in_data:
        excluded_missing = df_final[excluded_in_data].isna().sum().sum()
        print()
        print(f"Excluded variables (not imputed): {excluded_in_data}")
        print(f"Missing in excluded vars: {excluded_missing}")

    # Build the KNN mask: True where the cell was missing before and is now filled
    knn_mask = was_missing & df_final[numeric_cols].notna()

    return df_final, knn_mask


cmaster_imp, knn_mask = apply_knn_imputer(df_imputed, n_neighbors=5)


IMPUTATION STEP 2: KNN imputation (cross-country)

Missing before: 5,613
Missing after: 0
Filled: 5,613 cells

Excluded variables (not imputed): ['Landlocked', 'Subsoil_Metals_Dominant', 'Hydrocarbons_Dominant', 'Precious_Metals_Dominant', 'Total_Production_Value', 'Total_Reserves_Value', 'Total_Production', 'Total_Reserves']
Missing in excluded vars: 519


### 11a. KNN Reliance Score

Not all imputed values carry the same uncertainty. Cells filled by within-country linear interpolation (Step 10) are anchored by observed values on both sides of the gap, with a maximum span of 3 years. Cells filled by KNN (Step 11) have no observed data for that country-variable combination at all; the value is borrowed entirely from the 5 nearest countries in the feature space.

The KNN reliance score measures, for each country, what fraction of its total data cells were filled by KNN. Countries with high scores (above 10%) have a substantial share of synthetic data. This does not mean the values are wrong, but it does mean they are less grounded in that country's own observations. Downstream analysis in NB5 and NB6 should be aware of which countries sit in this category.


In [16]:
# For each country, compute the fraction of cells filled by KNN rather than
# observed or interpolated. Higher values mean more synthetic data.

_id_cols = ['Country Code', 'Country Name', 'Year']
_numeric_cols = list(knn_mask.columns)
total_cells_per_country = len(_numeric_cols) * cmaster_imp.groupby('Country Code')['Year'].count()

knn_cells_per_row = knn_mask.sum(axis=1)
knn_cells_df = cmaster_imp[['Country Code']].copy()
knn_cells_df['knn_cells'] = knn_cells_per_row.values
knn_per_country = knn_cells_df.groupby('Country Code')['knn_cells'].sum()

knn_reliance = pd.DataFrame({
    'total_cells': total_cells_per_country,
    'knn_filled': knn_per_country,
}).fillna(0)
knn_reliance['knn_reliance_pct'] = (
    100 * knn_reliance['knn_filled'] / knn_reliance['total_cells']
).round(2)
knn_reliance = knn_reliance.sort_values('knn_reliance_pct', ascending=False)

# Also compute per-variable breakdown for the worst countries
knn_by_country_var = cmaster_imp[['Country Code']].copy()
knn_by_country_var = pd.concat([knn_by_country_var.reset_index(drop=True),
                                knn_mask.reset_index(drop=True)], axis=1)
knn_var_totals = knn_by_country_var.groupby('Country Code')[_numeric_cols].sum()

print("=" * 70)
print("KNN RELIANCE SCORE BY COUNTRY")
print("=" * 70)
print()
print("Score = % of a country's (year x variable) cells filled by KNN.")
print("Higher = more synthetic data, less reliable for downstream analysis.")
print()
print("%-8s %10s %12s %8s" % ("Country", "KNN Cells", "Total Cells", "KNN %"))
print("-" * 42)
for cc, row in knn_reliance.iterrows():
    flag = ' <- HIGH' if row['knn_reliance_pct'] > 10 else ''
    print(f"{cc:<8} {int(row['knn_filled']):>10,} {int(row['total_cells']):>12,} "
          f"{row['knn_reliance_pct']:>7.1f}%{flag}")

# Countries with >10% KNN reliance: show which variables were most filled
high_reliance = knn_reliance[knn_reliance['knn_reliance_pct'] > 10].index.tolist()
if high_reliance:
    print()
    print("-" * 70)
    print("DETAIL: Variables most filled by KNN for high-reliance countries (>10%)")
    print("-" * 70)
    for cc in high_reliance:
        var_counts = knn_var_totals.loc[cc]
        top_vars = var_counts[var_counts > 0].sort_values(ascending=False).head(5)
        if len(top_vars) > 0:
            n_years = cmaster_imp[cmaster_imp['Country Code'] == cc]['Year'].nunique()
            pct = knn_reliance.loc[cc, 'knn_reliance_pct']
            print()
            print(f"  {cc} (KNN reliance: {pct:.1f}%):")
            for var, count in top_vars.items():
                pct_of_years = 100 * count / n_years
                vname = var[:50]
                print(f"    {vname:<50s} {int(count):>3} / {n_years} years ({pct_of_years:.0f}%)")
else:
    print("No countries exceed 10% KNN reliance.")

# Save the reliance scores
knn_reliance.to_csv("intermediary/knn_reliance_by_country.csv")
print()
print("  -> intermediary/knn_reliance_by_country.csv")

# Add knn_reliance_pct column to cmaster_imp for downstream notebooks
cmaster_imp = cmaster_imp.merge(
    knn_reliance[['knn_reliance_pct']],
    left_on='Country Code', right_index=True, how='left',
)


KNN RELIANCE SCORE BY COUNTRY

Score = % of a country's (year x variable) cells filled by KNN.
Higher = more synthetic data, less reliable for downstream analysis.

Country   KNN Cells  Total Cells    KNN %
------------------------------------------
LBY             187          900    20.8% <- HIGH
PNG             180          900    20.0% <- HIGH
BRN             167          900    18.6% <- HIGH
GNQ             162          900    18.0% <- HIGH
LBR             138          900    15.3% <- HIGH
UZB             120          900    13.3% <- HIGH
ARE             115          900    12.8% <- HIGH
TCD             100          900    11.1% <- HIGH
ARM              98          900    10.9% <- HIGH
TTO              97          900    10.8% <- HIGH
QAT              93          900    10.3% <- HIGH
MMR              86          900     9.6%
ZWE              85          900     9.4%
MWI              84          900     9.3%
VEN              82          900     9.1%
TUR              79          900

## 12. Save Intermediate Outputs

In [17]:
cmaster_imp.to_csv("intermediary/master_data_imputed.csv", index=False)
nrpv.to_csv("intermediary/NRCleanData.csv", index=False)

print("Intermediate outputs saved:")
print("  intermediary/master_data_imputed.csv")
print("  intermediary/NRCleanData.csv")

Intermediate outputs saved:
  intermediary/master_data_imputed.csv
  intermediary/NRCleanData.csv


## 13. Population Data and Final Master Files

World Bank population data (WDI) is reshaped from wide to long format and merged into both the master panel and the NR dataset. Population figures are needed to compute per-capita resource values in the descriptive statistics and regressions.

In [18]:
df_pop = pd.read_csv("rawdata/PopulationWDI.csv")

# Remove metadata rows (NaN country codes)
df_pop = df_pop[df_pop["Country Code"].notna()].copy()

id_cols = ["Country Name", "Country Code"]
year_cols = [col for col in df_pop.columns if "[YR" in col]

df_long = pd.melt(
    df_pop,
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="Year_raw",
    value_name="Population",
)

# Extract year from "1995 [YR1995]" format
df_long["Year"] = df_long["Year_raw"].str.extract(r"(\d{4})").astype(int)
df_long = df_long.drop(columns=["Year_raw"])

# Clean population values (handles ".." entries)
df_long["Population"] = pd.to_numeric(df_long["Population"], errors="coerce")
df_long = df_long[["Country Name", "Country Code", "Year", "Population"]]
df_long = df_long.sort_values(["Country Name", "Year"]).reset_index(drop=True)

print(f"Population data: {df_long.shape[0]:,} rows")
print(f"Year range: {df_long['Year'].min()}-{df_long['Year'].max()}")
print(f"Countries: {df_long['Country Code'].nunique()}")
print(f"Missing values: {df_long['Population'].isna().sum()}")

Population data: 8,040 rows
Year range: 1995-2024
Countries: 268
Missing values: 90


In [19]:
master = pd.read_csv("intermediary/master_data_imputed.csv")
master = master.merge(
    df_long[["Country Code", "Year", "Population"]],
    on=["Country Code", "Year"],
    how="left",
)

# Country Code already exists in nrpv from standardize_country mapping
nr = nrpv.merge(
    df_long[["Country Code", "Year", "Population"]],
    on=["Country Code", "Year"],
    how="left",
)

master.to_csv("intermediary/Master.csv", index=False)
nr.to_csv("intermediary/NaturalResource.csv", index=False)

print("Final outputs saved:")
print(f"  intermediary/Master.csv ({master.shape})")
print(f"  intermediary/NaturalResource.csv ({nr.shape})")
print(f"\nPopulation coverage in master: "
      f"{master['Population'].notna().sum()}/{len(master)} rows")

Final outputs saved:
  intermediary/Master.csv ((3275, 49))
  intermediary/NaturalResource.csv ((18912, 12))

Population coverage in master: 3275/3275 rows


In [20]:
master

,Country Code,Country Name,Year,Access to electricity (% of population),Adjusted savings: gross savings (% of GNI),Agriculture,Capital depreciation rate,Clientelism index,"Death rates, crude per 1000 people",Domestic credit to private sector (% of GDP),...,"Use of IMF credit (DOD, current US$)",Total_Production,Total_Reserves,Total_Production_Value,Total_Reserves_Value,Hydrocarbons_Dominant,Subsoil_Metals_Dominant,Precious_Metals_Dominant,knn_reliance_pct,Population
0,AGO,Angola,1995,60.999138,13.339001,21.904954,0.035367,0.827,18.700,16.341552,...,0.0,632.854603,0.0,3.930716e+09,0.0,1.0,0.0,0.0,7.22,13699778.0
1,AGO,Angola,1996,85.709963,14.343260,13.470317,0.038398,0.827,18.445,16.237726,...,0.0,715.976066,0.0,5.401322e+09,0.0,1.0,0.0,0.0,7.22,14170973.0
2,AGO,Angola,1997,32.568132,30.344651,8.057205,0.040405,0.827,18.184,6.468561,...,0.0,741.000000,0.0,5.163877e+09,0.0,1.0,0.0,0.0,7.22,14660413.0
3,AGO,Angola,1998,37.673555,33.879052,7.628807,0.040825,0.827,18.925,10.360313,...,0.0,730.849315,0.0,3.392030e+09,0.0,1.0,0.0,0.0,7.22,15159370.0
4,AGO,Angola,1999,32.604435,37.537803,7.054739,0.041406,0.827,18.518,7.277983,...,374707057.4,745.060274,0.0,4.886909e+09,0.0,1.0,0.0,0.0,7.22,15667235.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3270,ZWE,Zimbabwe,2015,33.700000,-11.438012,8.284831,0.054977,0.689,9.180,18.315744,...,463752752.4,13.474123,0.0,1.071219e+09,0.0,0.0,0.0,1.0,9.44,14399013.0
3271,ZWE,Zimbabwe,2016,42.500000,-5.670411,7.873585,0.056414,0.691,8.816,17.098496,...,455165191.5,11.761010,0.0,1.082211e+09,0.0,0.0,0.0,1.0,9.44,14600294.0
3272,ZWE,Zimbabwe,2017,44.000000,-3.024929,7.247882,0.057745,0.679,8.587,5.810054,...,482184792.8,12.592409,0.0,1.305091e+09,0.0,0.0,0.0,1.0,9.44,14812482.0
3273,ZWE,Zimbabwe,2018,45.400000,14.220834,7.319201,0.059023,0.698,8.321,5.830267,...,470895600.2,14.061096,0.0,1.885026e+09,0.0,0.0,0.0,1.0,9.44,15034452.0


In [21]:
print("=" * 70)
print("NB3: IMPUTATION PIPELINE SUMMARY")
print("=" * 70)

print()
print("--- Natural resource data ---")
print(f"Resources: {nrpv['Resource'].nunique()}")
print(f"Countries in NR data: {nrpv['Country'].nunique()}")
print(f"NR rows: {nrpv.shape[0]:,}")

print()
print("--- Sample selection ---")
print(f"Final sample: {cmaster_imp['Country Code'].nunique()} countries x {cmaster_imp['Year'].nunique()} years")
print(f"Variables: {cmaster_imp.shape[1] - 4} (excl. identifiers and knn_reliance_pct)")
print(f"Omitted countries: {len(omit_countries)}")

print()
print("--- Imputation ---")
remaining = cmaster_imp.select_dtypes(include='number').drop(columns=['knn_reliance_pct'], errors='ignore').isna().sum().sum()
print(f"Missing cells after full imputation: {remaining}")

print()
print("--- KNN reliance ---")
n_high = len(knn_reliance[knn_reliance['knn_reliance_pct'] > 10])
print(f"Countries with >10% KNN reliance: {n_high}")
if n_high > 0:
    for cc in knn_reliance[knn_reliance['knn_reliance_pct'] > 10].index:
        print(f"  {cc}: {knn_reliance.loc[cc, 'knn_reliance_pct']:.1f}%")

print()
print("--- Population merge ---")
print(f"Master.csv: {master.shape}")
print(f"Population coverage: {master['Population'].notna().sum()}/{len(master)} rows")

print()
print("Saved:")
print("  intermediary/master_data_imputed.csv")
print("  intermediary/NRCleanData.csv")
print("  intermediary/knn_reliance_by_country.csv")
print("  intermediary/Master.csv")
print("  intermediary/NaturalResource.csv")


NB3: IMPUTATION PIPELINE SUMMARY

--- Natural resource data ---
Resources: 21
Countries in NR data: 156
NR rows: 18,912

--- Sample selection ---
Final sample: 131 countries x 25 years
Variables: 44 (excl. identifiers and knn_reliance_pct)
Omitted countries: 24

--- Imputation ---
Missing cells after full imputation: 519

--- KNN reliance ---
Countries with >10% KNN reliance: 11
  LBY: 20.8%
  PNG: 20.0%
  BRN: 18.6%
  GNQ: 18.0%
  LBR: 15.3%
  UZB: 13.3%
  ARE: 12.8%
  TCD: 11.1%
  ARM: 10.9%
  TTO: 10.8%
  QAT: 10.3%

--- Population merge ---
Master.csv: (3275, 49)
Population coverage: 3275/3275 rows

Saved:
  intermediary/master_data_imputed.csv
  intermediary/NRCleanData.csv
  intermediary/knn_reliance_by_country.csv
  intermediary/Master.csv
  intermediary/NaturalResource.csv
